In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSV files (they are in current working directory)
df_metric = pd.read_csv('metric.csv', low_memory=False)
df_trace = pd.read_csv('trace.csv', low_memory=False)
df_log = pd.read_csv('log.csv', low_memory=False)
df_err = pd.read_csv('error_logs.csv', low_memory=False)

# Helper to convert timestamp column to timezone-aware datetimes (UTC->Asia/Shanghai)
def ts_min_max(df, ts_col='timestamp'):
    ts = pd.to_datetime(df[ts_col], unit='s', utc=True)
    ts_local = ts.dt.tz_convert(tz)
    return ts_local.min(), ts_local.max()

# 1) metric.csv aggregates
metric_kpi_counts = (
    df_metric['kpi_name']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'kpi_name', 'kpi_name': 'row_count'})
)
metric_kpi_counts = metric_kpi_counts.head(100)  # limit to top 100

metric_unique_cmdb_count = df_metric['cmdb_id'].nunique()
metric_min_ts, metric_max_ts = ts_min_max(df_metric, 'timestamp')
metric_meta = pd.DataFrame([{
    'unique_cmdb_count': metric_unique_cmdb_count,
    'min_timestamp': metric_min_ts,
    'max_timestamp': metric_max_ts
}])

# 2) trace.csv aggregates
trace_name_counts = (
    df_trace['trace_name']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'trace_name', 'trace_name': 'row_count'})
)
trace_name_counts = trace_name_counts.head(100)

trace_unique_cmdb_count = df_trace['cmdb_id'].nunique()
trace_min_ts, trace_max_ts = ts_min_max(df_trace, 'timestamp')
trace_meta = pd.DataFrame([{
    'unique_cmdb_count': trace_unique_cmdb_count,
    'min_timestamp': trace_min_ts,
    'max_timestamp': trace_max_ts
}])

# 3) log.csv aggregates
log_name_counts = (
    df_log['log_name']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'log_name', 'log_name': 'row_count'})
)
log_name_counts = log_name_counts.head(100)

log_unique_cmdb_count = df_log['cmdb_id'].nunique()
log_min_ts, log_max_ts = ts_min_max(df_log, 'timestamp')
log_meta = pd.DataFrame([{
    'unique_cmdb_count': log_unique_cmdb_count,
    'min_timestamp': log_min_ts,
    'max_timestamp': log_max_ts
}])

# 4) error_logs.csv aggregates
error_row_count = len(df_err)
error_unique_cmdb_count = df_err['cmdb_id'].nunique()
err_min_ts, err_max_ts = ts_min_max(df_err, 'timestamp')
error_meta = pd.DataFrame([{
    'row_count': error_row_count,
    'unique_cmdb_count': error_unique_cmdb_count,
    'min_timestamp': err_min_ts,
    'max_timestamp': err_max_ts
}])

# Combined top 10 most frequent cmdb_id across all four files
combined_cmdb = pd.concat([
    df_metric[['cmdb_id']],
    df_trace[['cmdb_id']],
    df_log[['cmdb_id']],
    df_err[['cmdb_id']]
], ignore_index=True)

combined_top10 = (
    combined_cmdb['cmdb_id']
    .dropna()
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'cmdb_id', 'cmdb_id': 'total_row_count'})
    .head(10)
)

# Display compact results (use variables rather than print)
metric_kpi_counts, metric_meta, trace_name_counts, trace_meta, log_name_counts, log_meta, error_meta, combined_top10

```
Out[1]:
```
result_summary = (
    "Summary of telemetry files (compact):\n\n"
    "1) metric.csv:\n"
    "- Top KPI name samples (showing top 100 separately): many container.* metrics appear; the highest per-KPI row_count observed in the top list is 1050.\n"
    "- Unique cmdb_id count: 59\n"
    "- Time range (UTC+8): 2022-03-21 00:00:00 to 2022-03-21 00:29:00\n\n"
    "2) trace.csv:\n"
    "- Top trace names include trace.from_frontend-0.duration_mean, duration_p95, error_rate, row_count (top entries with 577 rows) and other per-call metrics.\n"
    "- Unique cmdb_id count: 40\n"
    "- Time range (UTC+8): 2022-03-21 00:00:00 to 2022-03-21 00:29:00\n\n"
    "3) log.csv:\n"
    "- Top log names: log.error_count and log.row_count (each with 874 rows in the top list).\n"
    "- Unique cmdb_id count: 31\n"
    "- Time range (UTC+8): 2022-03-21 00:00:00 to 2022-03-21 00:29:00\n\n"
    "4) error_logs.csv:\n"
    "- Total rows: 540\n"
    "- Unique cmdb_id count: 3\n"
    "- Time range (UTC+8): 2022-03-21 00:00:05 to 2022-03-21 00:29:59\n\n"
    "Combined top 10 cmdb_id by total rows across all files:\n"
    "1) adservice: 10230\n"
    "2) adservice2: 9990\n"
    "3) frontend-0: 7637\n"
    "4) frontend-1: 7305\n"
    "5) frontend-2: 7178\n"
    "6) frontend2-0: 6076\n"
    "7) checkoutservice-0: 5450\n"
    "8) checkoutservice-1: 5248\n"
    "9) checkoutservice-2: 5206\n"
    "10) productcatalogservice-2: 4608\n\n"
    "Notes: Detailed per-name counts and the full top-100 lists are available in the previously returned DataFrames (metric_kpi_counts, trace_name_counts, log_name_counts, combined_top10)."
)
result_summary

The original code execution output of IPython Kernel is also provided below for reference:

(                                            row_count  count
0   container.node-5.container_network_receive_MB....   1050
1   container.node-5.container_network_receive_pac...   1050
2   container.node-5.container_network_transmit_pa...   1050
3   container.node-5.container_network_receive_err...   1050
4   container.node-5.container_network_transmit_er...   1050
..                                                ...    ...
95  container.node-6.container_network_transmit_pa...    210
96          container.node-6.container_spec_cpu_quota    210
97  container.node-6.container_cpu_cfs_throttled_s...    210
98              container.node-6.container_memory_rss    210
99  container.node-6.container_fs_write_seconds./d...    210

[100 rows x 2 columns],    unique_cmdb_count             min_timestamp             max_timestamp
0                 59 2022-03-21 00:00:00+08:00 2022-03-21 00:29:00+08:00,                                      row_count  count
0          trace.from_frontend-0.duration_mean    577
1           trace.from_frontend-0.duration_p95    577
2             trace.from_frontend-0.error_rate    577
3              trace.from_frontend-0.row_count    577
4              trace.from_frontend-2.row_count    555
..                                         ...    ...
95  trace.to_recommendationservice-1.row_count     90
96          trace.to_adservice-2.duration_mean     90
97           trace.to_adservice-2.duration_p95     90
98             trace.to_adservice-2.error_rate     90
99              trace.to_adservice-2.row_count     90

[100 rows x 2 columns],    unique_cmdb_count             min_timestamp             max_timestamp
0                 40 2022-03-21 00:00:00+08:00 2022-03-21 00:29:00+08:00,          row_count  count
0  log.error_count    874
1    log.row_count    874,    unique_cmdb_count             min_timestamp             max_timestamp
0                 31 2022-03-21 00:00:00+08:00 2022-03-21 00:29:00+08:00,    row_count  unique_cmdb_count             min_timestamp             max_timestamp
0        540                  3 2022-03-21 00:00:05+08:00 2022-03-21 00:29:59+08:00,            total_row_count  count
0                adservice  10230
1               adservice2   9990
2               frontend-0   7637
3               frontend-1   7305
4               frontend-2   7178
5              frontend2-0   6076
6        checkoutservice-0   5450
7        checkoutservice-1   5248
8        checkoutservice-2   5206
9  productcatalogservice-2   4608)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# reuse df_metric if already loaded; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', low_memory=False)

tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp as timezone-aware datetime in UTC+8
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95, P5, and full_series_count per (cmdb_id, kpi_name) using entire file
group_stats = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(full_series_count='count',
         P95=lambda x: x.quantile(0.95),
         P5=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Filter to incident window (2022-03-21 00:00 to 00:30 UTC+8) - [start, end)
start = pd.Timestamp('2022-03-21 00:00:00', tz=tz)
end = pd.Timestamp('2022-03-21 00:30:00', tz=tz)
df_window = df_metric[(df_metric['ts'] >= start) & (df_metric['ts'] < end)].copy()

# Merge global thresholds into windowed rows
if not df_window.empty:
    merged = df_window.merge(group_stats, on=['cmdb_id', 'kpi_name'], how='left')
    # flags for anomaly comparisons against global thresholds
    merged['ge_P95'] = merged['value'] >= merged['P95']
    merged['le_P5'] = merged['value'] <= merged['P5']

    # Aggregations within window per pair
    def earliest_ts_if_flag(ts_series, flag_series):
        if flag_series.any():
            return ts_series[flag_series].min()
        else:
            return pd.NaT

    grouped = merged.groupby(['cmdb_id', 'kpi_name'])
    window_aggs = grouped.agg(
        total_points_in_window=('value', 'size'),
        count_points_ge_P95=('ge_P95', 'sum'),
        count_points_le_P5=('le_P5', 'sum'),
        max_value_in_window=('value', 'max'),
        min_value_in_window=('value', 'min'),
    ).reset_index()

    # earliest timestamps computed with group-aware logic
    earliest_ge = grouped.apply(lambda g: earliest_ts_if_flag(g['ts'], g['ge_P95'])).reset_index()
    earliest_ge.columns = ['cmdb_id', 'kpi_name', 'earliest_timestamp_ge_P95']
    earliest_le = grouped.apply(lambda g: earliest_ts_if_flag(g['ts'], g['le_P5'])).reset_index()
    earliest_le.columns = ['cmdb_id', 'kpi_name', 'earliest_timestamp_le_P5']

    # Combine window aggregates and earliest timestamps
    window_summary = window_aggs.merge(earliest_ge, on=['cmdb_id','kpi_name'], how='left') \
                                .merge(earliest_le, on=['cmdb_id','kpi_name'], how='left')
else:
    # empty window summary
    window_summary = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','total_points_in_window','count_points_ge_P95','count_points_le_P5',
        'max_value_in_window','min_value_in_window','earliest_timestamp_ge_P95','earliest_timestamp_le_P5'
    ])

# 3) Combine full-series stats with window summary
final = group_stats.merge(window_summary, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for counts with 0 and keep earliest timestamps as NaT when missing
for c in ['total_points_in_window','count_points_ge_P95','count_points_le_P5']:
    if c in final.columns:
        final[c] = final[c].fillna(0).astype(int)
# Ensure min/max and earliest timestamps exist as columns
for c in ['max_value_in_window','min_value_in_window','earliest_timestamp_ge_P95','earliest_timestamp_le_P5']:
    if c not in final.columns:
        final[c] = pd.NA

# 4) Filter to only pairs that have anomalies in the window (count_ge>0 or count_le>0)
final['anomaly_count'] = final[['count_points_ge_P95','count_points_le_P5']].max(axis=1)
anomalous = final[final['anomaly_count'] > 0].copy()

# Rank by descending anomaly_count and limit to top 20
anomalous = anomalous.sort_values(by='anomaly_count', ascending=False).head(20)

# Select and order requested columns
result_cols = [
    'cmdb_id','kpi_name','full_series_count','P95','P5',
    'total_points_in_window','count_points_ge_P95','earliest_timestamp_ge_P95',
    'count_points_le_P5','earliest_timestamp_le_P5','max_value_in_window','min_value_in_window'
]
# Keep only existing columns (defensive)
result_cols = [c for c in result_cols if c in anomalous.columns]
final_result = anomalous[result_cols].reset_index(drop=True)

# Keep output compact: show up to 20 rows (already limited)
final_result

```
Out[2]:
```
result_summary = (
    "Summary (metric.csv, window 2022-03-21 00:00–00:30 UTC+8):\n\n"
    "Top pairs (up to 20) with anomalies: 20 rows found — all are for cmdb_id 'shippingservice2-0' "
    "and various container.node-5.* kpi_name values.\n\n"
    "For each listed (cmdb_id, kpi_name):\n"
    "- full_series_count = 30 (only 30 points exist in the file for these pairs)\n"
    "- P95 == P5 (the series is constant → P95 and P5 equal the observed value)\n"
    "- total_points_in_window = 30 (window contains all points)\n"
    "- count_points_ge_P95 = 30 and count_points_le_P5 = 30 (every point equals the threshold)\n"
    "- earliest timestamps for both >=P95 and <=P5 = 2022-03-21 00:00:00+08:00\n"
    "- max_value_in_window == min_value_in_window (the value is constant across the window)\n\n"
    "Interpretation: These KPIs are constant over the full series and the incident window; "
    "they appear as 'anomalous' only because the global P95 and P5 equal the observed constant value. "
    "No variable spike/drop anomalies were detected among the top 20 pairs."
)
result_summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  full_series_count           P95            P5  total_points_in_window  count_points_ge_P95 earliest_timestamp_ge_P95  count_points_le_P5  earliest_timestamp_le_P5  max_value_in_window  min_value_in_window
0   shippingservice2-0      container.node-5.container_memory_mapped_file                 30  4.915200e+05  4.915200e+05                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00         4.915200e+05         4.915200e+05
1   shippingservice2-0     container.node-5.container_memory_max_usage_MB                 30  1.280117e+02  1.280117e+02                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00         1.280117e+02         1.280117e+02
2   shippingservice2-0             container.node-5.container_memory_swap                 30  0.000000e+00  0.000000e+00                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00         0.000000e+00         0.000000e+00
3   shippingservice2-0  container.node-5.container_network_receive_err...                 30  0.000000e+00  0.000000e+00                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00         0.000000e+00         0.000000e+00
4   shippingservice2-0  container.node-5.container_fs_writes_merged./d...                 30  0.000000e+00  0.000000e+00                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00         0.000000e+00         0.000000e+00
5   shippingservice2-0            container.node-5.container_memory_cache                 30  1.913242e+07  1.913242e+07                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00         1.913242e+07         1.913242e+07
6   shippingservice2-0          container.node-5.container_memory_failcnt                 30  2.397029e+06  2.397029e+06                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00         2.397029e+06         2.397029e+06
7   shippingservice2-0  container.node-5.container_memory_failures.con...                 30  0.000000e+00  0.000000e+00                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00         0.000000e+00         0.000000e+00
8   shippingservice2-0  container.node-5.container_memory_failures.hie...                 30  0.000000e+00  0.000000e+00                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00         0.000000e+00         0.000000e+00
9   shippingservice2-0  container.node-5.container_fs_reads_merged./de...                 30  0.000000e+00  0.000000e+00                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00         0.000000e+00         0.000000e+00
10  shippingservice2-0  container.node-5.container_fs_sector_reads./de...                 30  0.000000e+00  0.000000e+00                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00         0.000000e+00         0.000000e+00
11  shippingservice2-0  container.node-5.container_fs_sector_writes./d...                 30  0.000000e+00  0.000000e+00                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00         0.000000e+00         0.000000e+00
12  shippingservice2-0  container.node-5.container_fs_write_seconds./d...                 30  0.000000e+00  0.000000e+00                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00         0.000000e+00         0.000000e+00
13  shippingservice2-0      container.node-5.container_fs_writes./dev/vda                 30  0.000000e+00  0.000000e+00                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00         0.000000e+00         0.000000e+00
14  shippingservice2-0     container.node-5.container_fs_writes./dev/vda1                 30  0.000000e+00  0.000000e+00                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00         0.000000e+00         0.000000e+00
15  shippingservice2-0   container.node-5.container_fs_writes_MB./dev/vda                 30  0.000000e+00  0.000000e+00                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00         0.000000e+00         0.000000e+00
16  shippingservice2-0  container.node-5.container_fs_io_current./dev/...                 30  0.000000e+00  0.000000e+00                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00         0.000000e+00         0.000000e+00
17  shippingservice2-0  container.node-5.container_fs_io_time_seconds....                 30  0.000000e+00  0.000000e+00                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00         0.000000e+00         0.000000e+00
18  shippingservice2-0  container.node-5.container_fs_io_time_weighted...                 30  0.000000e+00  0.000000e+00                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00         0.000000e+00         0.000000e+00
19  shippingservice2-0   container.node-5.container_fs_limit_MB./dev/vda1                 30  6.046307e+05  6.046307e+05                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00         6.046307e+05         6.046307e+05```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse df_trace if available, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv', low_memory=False)

# Ensure a timezone-aware timestamp column 'ts' in UTC+8
if 'ts' not in df_trace.columns:
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Global P95, P5, full-series count per (cmdb_id, trace_name)
group_stats_trace = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(
        full_series_count='count',
        P95=lambda x: x.quantile(0.95),
        P5=lambda x: x.quantile(0.05)
    )
    .reset_index()
)

# 2) Filter to incident window [2022-03-21 00:00, 2022-03-21 00:30) UTC+8
start = pd.Timestamp('2022-03-21 00:00:00', tz=tz)
end = pd.Timestamp('2022-03-21 00:30:00', tz=tz)
df_trace_window = df_trace[(df_trace['ts'] >= start) & (df_trace['ts'] < end)].copy()

# If window has rows, compute windowed aggregates using global thresholds
if not df_trace_window.empty:
    merged = df_trace_window.merge(group_stats_trace, on=['cmdb_id', 'trace_name'], how='left')
    merged['ge_P95'] = merged['value'] >= merged['P95']
    merged['le_P5'] = merged['value'] <= merged['P5']

    # Aggregations per pair within window
    window_aggs = (
        merged
        .groupby(['cmdb_id', 'trace_name'])
        .agg(
            total_points_in_window=('value', 'size'),
            count_points_ge_P95=('ge_P95', 'sum'),
            count_points_le_P5=('le_P5', 'sum'),
            max_value_in_window=('value', 'max'),
            min_value_in_window=('value', 'min')
        )
        .reset_index()
    )

    # Earliest timestamps where flags are true
    def first_ts_for_flag(df, flag_col):
        flagged = df[df[flag_col]]
        if flagged.empty:
            return pd.NaT
        return flagged['ts'].min()

    earliest_ge = (
        merged.groupby(['cmdb_id', 'trace_name'])
        .apply(lambda g: first_ts_for_flag(g, 'ge_P95'))
        .reset_index()
        .rename(columns={0: 'earliest_timestamp_ge_P95'})
    )
    earliest_le = (
        merged.groupby(['cmdb_id', 'trace_name'])
        .apply(lambda g: first_ts_for_flag(g, 'le_P5'))
        .reset_index()
        .rename(columns={0: 'earliest_timestamp_le_P5'})
    )

    window_summary = window_aggs.merge(earliest_ge, on=['cmdb_id','trace_name'], how='left') \
                                .merge(earliest_le, on=['cmdb_id','trace_name'], how='left')
else:
    window_summary = pd.DataFrame(columns=[
        'cmdb_id','trace_name','total_points_in_window','count_points_ge_P95','count_points_le_P5',
        'max_value_in_window','min_value_in_window','earliest_timestamp_ge_P95','earliest_timestamp_le_P5'
    ])

# 3) Combine global stats and window summary
final_trace = group_stats_trace.merge(window_summary, on=['cmdb_id','trace_name'], how='left')

# Normalize missing numeric counts to 0
for col in ['total_points_in_window','count_points_ge_P95','count_points_le_P5']:
    if col in final_trace.columns:
        final_trace[col] = final_trace[col].fillna(0).astype(int)

# Ensure timestamp cols exist
for col in ['earliest_timestamp_ge_P95','earliest_timestamp_le_P5']:
    if col not in final_trace.columns:
        final_trace[col] = pd.NaT

# 4) Filter to pairs with anomalies in window and rank by max(count_ge, count_le)
final_trace['anomaly_count'] = final_trace[['count_points_ge_P95','count_points_le_P5']].max(axis=1)
anomalous_trace = final_trace[final_trace['anomaly_count'] > 0].copy()
anomalous_trace = anomalous_trace.sort_values(by='anomaly_count', ascending=False).head(20)

# Select and order requested columns (defensive: include only existing)
cols = [
    'cmdb_id','trace_name','full_series_count','P95','P5',
    'total_points_in_window','count_points_ge_P95','earliest_timestamp_ge_P95',
    'count_points_le_P5','earliest_timestamp_le_P5','max_value_in_window','min_value_in_window'
]
cols = [c for c in cols if c in anomalous_trace.columns]
final_result = anomalous_trace[cols].reset_index(drop=True)

# Display compact result (up to 20 rows)
final_result

```
Out[3]:
```

```python
result_summary = (
    "Summary (trace.csv, window 2022-03-21 00:00–00:30 UTC+8):\n\n"
    "- Found up to 20 (cmdb_id, trace_name) pairs with 'anomalies' based on global P95/P5. "
    "All listed pairs have full_series_count = 30 and P95 = P5 = 0.\n\n"
    "- For each listed pair:\n"
    "  • total_points_in_window = 30 (window contains all points)\n"
    "  • count_points_ge_P95 = 30 and count_points_le_P5 = 30 (every point equals the threshold)\n"
    "  • earliest timestamps for >=P95 and <=P5 = 2022-03-21 00:00:00+08:00\n"
    "  • max_value_in_window = min_value_in_window = 0.0 (series is constant)\n\n"
    "Interpretation: These trace error_rate series are constant at 0 across the full series and the incident window. "
    "They appear flagged only because global P95 and P5 equal the observed constant value — no true spikes or drops "
    "were detected among the top 20 pairs."
)
result_summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                        trace_name  full_series_count  P95   P5  total_points_in_window  count_points_ge_P95 earliest_timestamp_ge_P95  count_points_le_P5  earliest_timestamp_le_P5  max_value_in_window  min_value_in_window
0         adservice-0  trace.from_frontend-2.error_rate                 30  0.0  0.0                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00                  0.0                  0.0
1         adservice-0  trace.from_frontend-1.error_rate                 30  0.0  0.0                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00                  0.0                  0.0
2         adservice-1  trace.from_frontend-1.error_rate                 30  0.0  0.0                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00                  0.0                  0.0
3         adservice-1  trace.from_frontend-0.error_rate                 30  0.0  0.0                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00                  0.0                  0.0
4         adservice-0  trace.from_frontend-0.error_rate                 30  0.0  0.0                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00                  0.0                  0.0
5         adservice-2  trace.from_frontend-0.error_rate                 30  0.0  0.0                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00                  0.0                  0.0
6         adservice-1  trace.from_frontend-2.error_rate                 30  0.0  0.0                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00                  0.0                  0.0
7         adservice-2  trace.from_frontend-2.error_rate                 30  0.0  0.0                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00                  0.0                  0.0
8         adservice-2  trace.from_frontend-1.error_rate                 30  0.0  0.0                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00                  0.0                  0.0
9   currencyservice-1  trace.from_frontend-1.error_rate                 30  0.0  0.0                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00                  0.0                  0.0
10  currencyservice-0  trace.from_frontend-2.error_rate                 30  0.0  0.0                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00                  0.0                  0.0
11  currencyservice-0  trace.from_frontend-0.error_rate                 30  0.0  0.0                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00                  0.0                  0.0
12  currencyservice-0  trace.from_frontend-1.error_rate                 30  0.0  0.0                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00                  0.0                  0.0
13  currencyservice-2  trace.from_frontend-0.error_rate                 30  0.0  0.0                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00                  0.0                  0.0
14  currencyservice-2  trace.from_frontend-1.error_rate                 30  0.0  0.0                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00                  0.0                  0.0
15  currencyservice-1  trace.from_frontend-2.error_rate                 30  0.0  0.0                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00                  0.0                  0.0
16  currencyservice-1  trace.from_frontend-0.error_rate                 30  0.0  0.0                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00                  0.0                  0.0
17         frontend-0             trace.self.error_rate                 30  0.0  0.0                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00                  0.0                  0.0
18         frontend-0   trace.to_adservice-0.error_rate                 30  0.0  0.0                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00                  0.0                  0.0
19         frontend-0        trace.from_root.error_rate                 30  0.0  0.0                      30                   30 2022-03-21 00:00:00+08:00                  30 2022-03-21 00:00:00+08:00                  0.0                  0.0```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse previously loaded dataframes if present, otherwise load
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv', low_memory=False)
try:
    df_err
except NameError:
    df_err = pd.read_csv('error_logs.csv', low_memory=False)

# Ensure timezone-aware timestamp columns for both dataframes (UTC -> convert to Asia/Shanghai)
if 'ts' not in df_log.columns:
    df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
if 'ts' not in df_err.columns:
    df_err['ts'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Incident window (inclusive start, exclusive end) in UTC+8
start = pd.Timestamp('2022-03-21 00:00:00', tz=tz)
end = pd.Timestamp('2022-03-21 00:30:00', tz=tz)

# ------------------------
# 1) log.csv analysis
# ------------------------
# Global thresholds (P95, P5) and full_series_count per (cmdb_id, log_name)
log_group_stats = (
    df_log
    .groupby(['cmdb_id', 'log_name'])['value']
    .agg(
        full_series_count='count',
        P95=lambda x: x.quantile(0.95),
        P5=lambda x: x.quantile(0.05)
    )
    .reset_index()
)

# Filter to window
df_log_window = df_log[(df_log['ts'] >= start) & (df_log['ts'] < end)].copy()

if not df_log_window.empty:
    # Merge thresholds into windowed rows
    merged_log = df_log_window.merge(log_group_stats, on=['cmdb_id', 'log_name'], how='left')
    # Flags for comparisons (strict > for P95 and < for P5 as requested)
    merged_log['gt_P95'] = merged_log['value'] > merged_log['P95']
    merged_log['lt_P5'] = merged_log['value'] < merged_log['P5']

    # Windowed aggregations
    log_window_aggs = (
        merged_log
        .groupby(['cmdb_id', 'log_name'])
        .agg(
            total_points_in_window=('value', 'size'),
            sum_value_in_window=('value', 'sum'),
            count_values_gt_P95=('gt_P95', 'sum'),
            count_values_lt_P5=('lt_P5', 'sum'),
            max_value_in_window=('value', 'max'),
            min_value_in_window=('value', 'min')
        )
        .reset_index()
    )

    # Earliest timestamps for the conditions
    def first_ts_where(df, cond_col):
        flagged = df[df[cond_col]]
        return flagged['ts'].min() if not flagged.empty else pd.NaT

    earliest_gt = (
        merged_log.groupby(['cmdb_id','log_name'])
        .apply(lambda g: first_ts_where(g, 'gt_P95'))
        .reset_index()
        .rename(columns={0: 'earliest_timestamp_value_gt_P95'})
    )
    earliest_lt = (
        merged_log.groupby(['cmdb_id','log_name'])
        .apply(lambda g: first_ts_where(g, 'lt_P5'))
        .reset_index()
        .rename(columns={0: 'earliest_timestamp_value_lt_P5'})
    )

    # Combine window aggregates with earliest timestamps
    log_window_summary = (log_window_aggs
                          .merge(earliest_gt, on=['cmdb_id','log_name'], how='left')
                          .merge(earliest_lt, on=['cmdb_id','log_name'], how='left'))
else:
    # empty window summary
    log_window_summary = pd.DataFrame(columns=[
        'cmdb_id','log_name','total_points_in_window','sum_value_in_window',
        'count_values_gt_P95','count_values_lt_P5',
        'max_value_in_window','min_value_in_window',
        'earliest_timestamp_value_gt_P95','earliest_timestamp_value_lt_P5'
    ])

# Combine global stats with window summary
log_final = log_group_stats.merge(log_window_summary, on=['cmdb_id','log_name'], how='left')

# Fill numeric NaNs with 0 for counts/sums to keep table compact
for col in ['total_points_in_window','sum_value_in_window','count_values_gt_P95','count_values_lt_P5']:
    if col in log_final.columns:
        log_final[col] = log_final[col].fillna(0)
# Ensure earliest timestamp cols exist
for col in ['earliest_timestamp_value_gt_P95','earliest_timestamp_value_lt_P5']:
    if col not in log_final.columns:
        log_final[col] = pd.NaT

# Select top 20 by sum_value_in_window descending (if sum missing treat as 0)
log_final['sum_value_in_window'] = log_final['sum_value_in_window'].fillna(0)
log_summary_top20 = log_final.sort_values(by='sum_value_in_window', ascending=False).head(20)

# Select and order requested columns (only existing ones)
log_cols = [
    'cmdb_id','log_name','full_series_count','P95','P5',
    'total_points_in_window','sum_value_in_window','count_values_gt_P95','earliest_timestamp_value_gt_P95',
    'count_values_lt_P5','earliest_timestamp_value_lt_P5'
]
log_cols = [c for c in log_cols if c in log_summary_top20.columns]
log_summary_top20 = log_summary_top20[log_cols].reset_index(drop=True)

# ------------------------
# 2) error_logs.csv analysis
# ------------------------
# Filter error logs to window
df_err_window = df_err[(df_err['ts'] >= start) & (df_err['ts'] < end)].copy()

# Compact summary: total rows in window and unique cmdb count in window
error_window_meta = pd.DataFrame([{
    'total_rows_in_window': len(df_err_window),
    'unique_cmdb_count_in_window': int(df_err_window['cmdb_id'].nunique())
}])

# Top 10 cmdb_id by count in window with earliest and latest timestamps
if not df_err_window.empty:
    err_group = (
        df_err_window.groupby('cmdb_id')
        .agg(
            count_in_window=('message','size'),
            earliest_timestamp_in_window=('ts','min'),
            latest_timestamp_in_window=('ts','max')
        )
        .reset_index()
    )
    top10_err_cmdb = err_group.sort_values(by='count_in_window', ascending=False).head(10).reset_index(drop=True)
else:
    top10_err_cmdb = pd.DataFrame(columns=['cmdb_id','count_in_window','earliest_timestamp_in_window','latest_timestamp_in_window'])

# For top 3 cmdb_id by count, get up to 20 unique message samples and their counts per component
top3_cmdbs = top10_err_cmdb['cmdb_id'].head(3).tolist()
top3_err_messages = {}
for cmdb in top3_cmdbs:
    msgs = (df_err_window[df_err_window['cmdb_id'] == cmdb]
            .groupby('message')['message']
            .agg(count='count')
            .reset_index()
            .sort_values(by='count', ascending=False)
            .head(20))
    # keep as dataframe per cmdb_id for compactness
    top3_err_messages[cmdb] = msgs

# Prepare final compact outputs (bounded)
# - log_summary_top20 (top 20 rows)
# - error_window_meta (1-row summary)
# - top10_err_cmdb (top 10 rows)
# - top3_err_messages (dict of up to 3 small DataFrames with up to 20 rows each)

log_summary_top20, error_window_meta, top10_err_cmdb, top3_err_messages

```
Out[4]:
```
Summary (compact, plain English):

1) log.csv (top 20 pairs by sum_value_in_window)
- All top entries are the metric log.row_count for various services. Each pair has full_series_count = 30 (data covers 30 minutes).
- Highlights (top 5 by sum_value_in_window):
  - frontend-0: sum_value_in_window = 17615, P95=711.15, P5=509.25, total_points_in_window=30, count_values_gt_P95=2 (first at 2022-03-21 00:03+08:00), count_values_lt_P5=2 (first at 00:12+08:00).
  - frontend-2: sum = 12537, P95=471.50, P5=326.35, count_gt_P95=2 (first 00:00), count_lt_P5=2 (first 00:12).
  - adservice-0: sum = 12246, P95=517.10, P5=330.80, count_gt_P95=2 (first 00:00), count_lt_P5=2 (first 00:03).
  - frontend-1: sum = 12024, P95=455.65, P5=325.60, count_gt_P95=2 (first 00:22), count_lt_P5=2 (first 00:08).
  - productcatalogservice-1: sum = 10255, P95=388.10, P5=294.15, count_gt_P95=2 (first 00:01), count_lt_P5=2 (first 00:00).
- Pattern: every listed pair has 30 points in-window. A small number of minutes (1–2) exceed the global P95 or fall below P5 for each service — i.e., brief spikes/dips in log row counts at specific minutes.

2) error_logs.csv (incident window 2022-03-21 00:00–00:30 UTC+8)
- Window summary: total_rows_in_window = 540; unique cmdb_id in window = 3.
- Top cmdb_id by error-row count (top 3 of top 10):
  1. frontend-0: count_in_window = 229, earliest = 2022-03-21 00:00:11+08:00, latest = 2022-03-21 00:29:59+08:00
  2. frontend-2: count_in_window = 158, earliest = 2022-03-21 00:00:05+08:00, latest = 2022-03-21 00:29:42+08:00
  3. frontend-1: count_in_window = 153, earliest = 2022-03-21 00:00:12+08:00, latest = 2022-03-21 00:29:54+08:00
- Message samples for top 3 components: each top component’s error rows are dominated by a single repeated message (same text) — the sample message appears for all counted rows per component (counts shown match the component counts above). Up to 20 unique messages per component were examined; the top message accounts for all reported error rows in each front-end component.

Interpretation / short conclusions:
- Log activity (log.row_count) shows brief spikes/dips (1–2 minutes per service exceed P95 or fall below P5) during the window across many services; these are short-lived anomalies in row count.
- Error logs are concentrated on frontend nodes (frontend-0, frontend-2, frontend-1) with hundreds of error rows in the window, mostly repeating the same warning message — this indicates frontend components were the main source of logged errors during the incident window.

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id       log_name  full_series_count     P95      P5  total_points_in_window  sum_value_in_window  count_values_gt_P95 earliest_timestamp_value_gt_P95  count_values_lt_P5 earliest_timestamp_value_lt_P5
0                frontend-0  log.row_count                 30  711.15  509.25                      30              17615.0                    2       2022-03-21 00:03:00+08:00                   2      2022-03-21 00:12:00+08:00
1                frontend-2  log.row_count                 30  471.50  326.35                      30              12537.0                    2       2022-03-21 00:00:00+08:00                   2      2022-03-21 00:12:00+08:00
2               adservice-0  log.row_count                 30  517.10  330.80                      30              12246.0                    2       2022-03-21 00:00:00+08:00                   2      2022-03-21 00:03:00+08:00
3                frontend-1  log.row_count                 30  455.65  325.60                      30              12024.0                    2       2022-03-21 00:22:00+08:00                   2      2022-03-21 00:08:00+08:00
4   productcatalogservice-1  log.row_count                 30  388.10  294.15                      30              10255.0                    2       2022-03-21 00:01:00+08:00                   2      2022-03-21 00:00:00+08:00
5   productcatalogservice-0  log.row_count                 30  393.85  296.80                      30              10242.0                    2       2022-03-21 00:01:00+08:00                   2      2022-03-21 00:00:00+08:00
6   productcatalogservice-2  log.row_count                 30  397.10  290.35                      30              10235.0                    2       2022-03-21 00:01:00+08:00                   2      2022-03-21 00:00:00+08:00
7             cartservice-1  log.row_count                 30  329.00  249.35                      30               8573.0                    1       2022-03-21 00:15:00+08:00                   2      2022-03-21 00:12:00+08:00
8             cartservice-0  log.row_count                 30  328.00  246.05                      30               8564.0                    1       2022-03-21 00:15:00+08:00                   2      2022-03-21 00:10:00+08:00
9             cartservice-2  log.row_count                 30  325.50  240.80                      30               8554.0                    2       2022-03-21 00:01:00+08:00                   2      2022-03-21 00:12:00+08:00
10        currencyservice-0  log.row_count                 30  282.65  215.25                      30               7505.0                    2       2022-03-21 00:00:00+08:00                   2      2022-03-21 00:12:00+08:00
11        currencyservice-2  log.row_count                 30  282.75  217.90                      30               7484.0                    2       2022-03-21 00:00:00+08:00                   2      2022-03-21 00:12:00+08:00
12        currencyservice-1  log.row_count                 30  282.30  216.60                      30               7470.0                    2       2022-03-21 00:00:00+08:00                   2      2022-03-21 00:12:00+08:00
13  recommendationservice-1  log.row_count                 30   84.55   65.35                      30               2255.0                    2       2022-03-21 00:01:00+08:00                   2      2022-03-21 00:00:00+08:00
14  recommendationservice-0  log.row_count                 30   86.00   64.00                      30               2253.0                    1       2022-03-21 00:01:00+08:00                   1      2022-03-21 00:18:00+08:00
15  recommendationservice-2  log.row_count                 30   85.00   64.45                      30               2247.0                    1       2022-03-21 00:01:00+08:00                   2      2022-03-21 00:12:00+08:00
16              adservice-1  log.row_count                 30   46.00   30.00                      30               1084.0                    0                             NaT                   1      2022-03-21 00:09:00+08:00
17              adservice-2  log.row_count                 30   44.20   30.00                      30               1082.0                    2       2022-03-21 00:00:00+08:00                   1      2022-03-21 00:09:00+08:00
18        shippingservice-2  log.row_count                 30   45.55   18.00                      30                957.0                    2       2022-03-21 00:13:00+08:00                   1      2022-03-21 00:00:00+08:00
19        shippingservice-1  log.row_count                 30   50.20   14.00                      30                956.0                    2       2022-03-21 00:03:00+08:00                   0                            NaT,    total_rows_in_window  unique_cmdb_count_in_window
0                   540                            3,       cmdb_id  count_in_window earliest_timestamp_in_window latest_timestamp_in_window
0  frontend-0              229    2022-03-21 00:00:11+08:00  2022-03-21 00:29:59+08:00
1  frontend-2              158    2022-03-21 00:00:05+08:00  2022-03-21 00:29:42+08:00
2  frontend-1              153    2022-03-21 00:00:12+08:00  2022-03-21 00:29:54+08:00, {'frontend-0':                                              message  count
0  severity: warning, message: failed to retrieve...    229, 'frontend-2':                                              message  count
0  severity: warning, message: failed to retrieve...    158, 'frontend-1':                                              message  count
0  severity: warning, message: failed to retrieve...    153})```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# reuse existing variables (df_log_window, df_err_window, log_group_stats) if present
tz = pytz.timezone('Asia/Shanghai')

try:
    df_log_window
except NameError:
    # load df_log and create window if missing
    df_log = pd.read_csv('log.csv', low_memory=False)
    df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    start = pd.Timestamp('2022-03-21 00:00:00', tz=tz)
    end = pd.Timestamp('2022-03-21 00:30:00', tz=tz)
    df_log_window = df_log[(df_log['ts'] >= start) & (df_log['ts'] < end)].copy()

try:
    df_err_window
except NameError:
    df_err = pd.read_csv('error_logs.csv', low_memory=False)
    df_err['ts'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    start = pd.Timestamp('2022-03-21 00:00:00', tz=tz)
    end = pd.Timestamp('2022-03-21 00:30:00', tz=tz)
    df_err_window = df_err[(df_err['ts'] >= start) & (df_err['ts'] < end)].copy()

try:
    log_group_stats
except NameError:
    # compute global P95 per (cmdb_id, log_name)
    df_log = pd.read_csv('log.csv', low_memory=False)
    df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    log_group_stats = (
        df_log
        .groupby(['cmdb_id', 'log_name'])['value']
        .agg(full_series_count='count', P95=lambda x: x.quantile(0.95), P5=lambda x: x.quantile(0.05))
        .reset_index()
    )

# INCIDENT WINDOW MINUTES (ensure minute column)
df_log_window['minute'] = df_log_window['ts'].dt.floor('T')
df_err_window['minute'] = df_err_window['ts'].dt.floor('T')

# 1) log.csv per-minute aggregation focusing on log.row_count
# Use global P95 computed per (cmdb_id, log_name)
log_p95_rowcount = log_group_stats[log_group_stats['log_name'] == 'log.row_count'][['cmdb_id','P95']].copy()

# Merge P95 into window rows for log.row_count only
log_rowcount_window = df_log_window[df_log_window['log_name'] == 'log.row_count'].merge(
    log_p95_rowcount, on='cmdb_id', how='left'
)

# Compute boolean exceed per row (value > P95)
log_rowcount_window['exceeds_P95'] = log_rowcount_window['value'] > log_rowcount_window['P95']

# Per-minute aggregation:
# - number_of_components_where_log.row_count > that component's P95 (unique cmdb_id count where exceeds)
# - list_of_up_to_20_cmdb_ids_exceeding_P95
# - sum_log_row_count_across_all_components for that minute
def agg_minute_log(g):
    # unique cmdb ids exceeding
    exceeding_cmdbs = g.loc[g['exceeds_P95'], 'cmdb_id'].dropna().unique().tolist()
    exceeding_cmdbs_limited = exceeding_cmdbs[:20]
    return pd.Series({
        'num_components_exceeding_P95': len(exceeding_cmdbs),
        'exceeding_cmdb_ids_list': ','.join(exceeding_cmdbs_limited),
        'sum_log_row_count': g['value'].sum()
    })

log_minute_agg = log_rowcount_window.groupby('minute').apply(agg_minute_log).reset_index()

# Ensure minutes within the incident window are represented even if zero (optional)
start = pd.Timestamp('2022-03-21 00:00:00', tz=tz)
end = pd.Timestamp('2022-03-21 00:30:00', tz=tz)
all_minutes = pd.DataFrame({'minute': pd.date_range(start, end - pd.Timedelta(minutes=1), freq='T', tz=tz)})
log_minute_agg = all_minutes.merge(log_minute_agg, on='minute', how='left').fillna({
    'num_components_exceeding_P95': 0,
    'exceeding_cmdb_ids_list': '',
    'sum_log_row_count': 0
})
# convert counts to int
log_minute_agg['num_components_exceeding_P95'] = log_minute_agg['num_components_exceeding_P95'].astype(int)
log_minute_agg['sum_log_row_count'] = log_minute_agg['sum_log_row_count'].astype(float)

# Rank minutes: primary by num_components_exceeding_P95 desc, secondary by sum_log_row_count desc
log_top10_minutes = log_minute_agg.sort_values(
    by=['num_components_exceeding_P95','sum_log_row_count'],
    ascending=[False, False]
).head(10).reset_index(drop=True)

# Keep compact columns and limit lists (already limited to 20)
log_top10_minutes = log_top10_minutes[['minute','num_components_exceeding_P95','exceeding_cmdb_ids_list','sum_log_row_count']]

# 2) error_logs.csv per-minute aggregation
if not df_err_window.empty:
    # per-minute total counts, earliest and latest timestamp
    err_minute_basic = (
        df_err_window
        .groupby('minute')
        .agg(total_error_rows=('message','size'),
             earliest_error_in_minute=('ts','min'),
             latest_error_in_minute=('ts','max'))
        .reset_index()
    )

    # per-minute top 5 cmdb_id counts
    err_minute_cmdb_counts = (
        df_err_window
        .groupby(['minute','cmdb_id'])
        .size()
        .reset_index(name='count')
    )

    # For each minute, get top 5 cmdb_id:count pairs as 'cmdb:count' joined string
    def top5_cmdbs_for_minute(minute_df):
        top5 = minute_df.sort_values(by='count', ascending=False).head(5)
        pairs = (top5['cmdb_id'].astype(str) + ':' + top5['count'].astype(str)).tolist()
        return ','.join(pairs)

    top5_list = err_minute_cmdb_counts.groupby('minute').apply(top5_cmdbs_for_minute).reset_index()
    top5_list.columns = ['minute','top_5_cmdb_counts']

    err_minute_agg = err_minute_basic.merge(top5_list, on='minute', how='left')
else:
    err_minute_agg = pd.DataFrame(columns=['minute','total_error_rows','earliest_error_in_minute','latest_error_in_minute','top_5_cmdb_counts'])

# Ensure minutes within incident window are present (fill zeros)
err_minute_agg = all_minutes.merge(err_minute_agg, on='minute', how='left').fillna({
    'total_error_rows': 0,
    'earliest_error_in_minute': pd.NaT,
    'latest_error_in_minute': pd.NaT,
    'top_5_cmdb_counts': ''
})
err_minute_agg['total_error_rows'] = err_minute_agg['total_error_rows'].astype(int)

# Top 10 minutes ranked by total_error_rows desc
err_top10_minutes = err_minute_agg.sort_values(by='total_error_rows', ascending=False).head(10).reset_index(drop=True)
err_top10_minutes = err_top10_minutes[['minute','total_error_rows','top_5_cmdb_counts','earliest_error_in_minute','latest_error_in_minute']]

# 3) earliest two distinct minutes that appear as peaks in both tables (intersection of top-10 minutes)
# Use minute values present in both top-10 lists
set_log_top = set(log_top10_minutes['minute'].tolist())
set_err_top = set(err_top10_minutes['minute'].tolist())
common_minutes = sorted(list(set_log_top & set_err_top))

if len(common_minutes) >= 2:
    overlap_minutes = pd.DataFrame({'overlap_minute': common_minutes[:2]})
elif len(common_minutes) == 1:
    # one common minute -> include it and the next earliest top minute from error table that is different
    first = common_minutes[0]
    # find top minutes from err_top10_minutes not equal to first
    other = [m for m in err_top10_minutes['minute'].tolist() if m != first]
    picks = [first] + other[:1]
    overlap_minutes = pd.DataFrame({'overlap_minute': picks})
else:
    # no overlap -> return top two minutes from error_logs table
    top_err_two = err_top10_minutes['minute'].dropna().unique().tolist()[:2]
    overlap_minutes = pd.DataFrame({'overlap_minute': top_err_two})

# Final compact outputs:
# - log_top10_minutes (top 10 minutes by number of components exceeding P95)
# - err_top10_minutes (top 10 minutes by total_error_rows)
# - overlap_minutes (earliest two distinct overlapping minutes or fallback)
log_top10_minutes, err_top10_minutes, overlap_minutes

```
Out[5]:
```
result = (
    "Summary (incident window 2022-03-21 00:00–00:30 UTC+8):\n\n"
    "1) log.csv (per-minute peaks vs per-component P95)\n"
    "- Peak minute: 00:15 — 10 components exceeded their P95; sum log.row_count = 5,509.\n"
    "- Other notable minutes: 00:01 (7 comps, sum=5,390), 00:00 (6 comps, sum=4,933), 00:19 (3 comps, sum=5,541), 00:22 (3 comps, sum=5,236).\n"
    "- The lists of components exceeding P95 are per-minute (limited to 20 in the output); examples include cartservice-0/1/2, productcatalogservice-*, currencyservice-*, adservice-*, frontend-*, etc.\n\n"
    "2) error_logs.csv (per-minute error counts)\n"
    "- Peak minute: 00:16 with 24 error rows (top contributors: frontend-0:9, frontend-2:8, frontend-1:7). Earliest error in that minute: 00:16:01, latest: 00:16:59.\n"
    "- Next peaks: 00:00 (23 errors; frontends: frontend-0:10, frontend-1:7, frontend-2:6; 00:00:05–00:00:57) and 00:02 (23 errors; frontends: frontend-0:10, frontend-2:7, frontend-1:6; 00:02:10–00:02:58).\n"
    "- Top-5 cmdb_id:count per minute are shown in the output (limited to 5 entries per minute).\n\n"
    "3) Overlap between the two peak lists\n"
    "- The earliest two minutes that are peaks in both tables are 00:00 and 00:02 (UTC+8). These minutes are present among the top-10 for both log.exceedance and error rows.\n\n"
    "Interpretation:\n"
    "- Log-row-count anomalies (most components exceeding their historical P95) concentrate around 00:15 and early-minute points like 00:00–00:01.\n"
    "- Error activity is concentrated in the front-end components, peaking at 00:16, with strong error bursts at 00:00 and 00:02.\n"
    "- Minutes 00:00 and 00:02 are common peaks in both analyses and are likely good starting points for correlated investigation (they coincide with elevated error activity and multiple components exceeding log P95)."
)
result

The original code execution output of IPython Kernel is also provided below for reference:

(                     minute  num_components_exceeding_P95                            exceeding_cmdb_ids_list  sum_log_row_count
0 2022-03-21 00:15:00+08:00                            10  cartservice-0,cartservice-1,currencyservice-1,...             5509.0
1 2022-03-21 00:01:00+08:00                             7  cartservice-2,productcatalogservice-0,productc...             5390.0
2 2022-03-21 00:00:00+08:00                             6  adservice-0,adservice-2,currencyservice-0,curr...             4933.0
3 2022-03-21 00:19:00+08:00                             3  cartservice-2,emailservice-1,productcatalogser...             5541.0
4 2022-03-21 00:22:00+08:00                             3      currencyservice-0,frontend-1,paymentservice-1             5236.0
5 2022-03-21 00:03:00+08:00                             3  frontend-0,productcatalogservice-0,shippingser...             5096.0
6 2022-03-21 00:29:00+08:00                             2                    emailservice-2,paymentservice-2             5052.0
7 2022-03-21 00:02:00+08:00                             2                checkoutservice-2,currencyservice-2             5049.0
8 2022-03-21 00:16:00+08:00                             2                            adservice-0,adservice-2             5042.0
9 2022-03-21 00:23:00+08:00                             2                    emailservice-0,paymentservice-2             4870.0,                      minute  total_error_rows                        top_5_cmdb_counts  earliest_error_in_minute    latest_error_in_minute
0 2022-03-21 00:16:00+08:00                24   frontend-0:9,frontend-2:8,frontend-1:7 2022-03-21 00:16:01+08:00 2022-03-21 00:16:59+08:00
1 2022-03-21 00:00:00+08:00                23  frontend-0:10,frontend-1:7,frontend-2:6 2022-03-21 00:00:05+08:00 2022-03-21 00:00:57+08:00
2 2022-03-21 00:02:00+08:00                23  frontend-0:10,frontend-2:7,frontend-1:6 2022-03-21 00:02:10+08:00 2022-03-21 00:02:58+08:00
3 2022-03-21 00:04:00+08:00                22   frontend-0:9,frontend-2:7,frontend-1:6 2022-03-21 00:04:01+08:00 2022-03-21 00:04:55+08:00
4 2022-03-21 00:10:00+08:00                21   frontend-0:8,frontend-2:8,frontend-1:5 2022-03-21 00:10:01+08:00 2022-03-21 00:10:57+08:00
5 2022-03-21 00:06:00+08:00                20   frontend-2:8,frontend-0:6,frontend-1:6 2022-03-21 00:06:03+08:00 2022-03-21 00:06:59+08:00
6 2022-03-21 00:22:00+08:00                20   frontend-1:9,frontend-0:6,frontend-2:5 2022-03-21 00:22:00+08:00 2022-03-21 00:22:57+08:00
7 2022-03-21 00:26:00+08:00                19   frontend-1:7,frontend-0:6,frontend-2:6 2022-03-21 00:26:00+08:00 2022-03-21 00:26:59+08:00
8 2022-03-21 00:14:00+08:00                19   frontend-0:7,frontend-1:6,frontend-2:6 2022-03-21 00:14:09+08:00 2022-03-21 00:14:59+08:00
9 2022-03-21 00:08:00+08:00                19   frontend-2:8,frontend-0:6,frontend-1:5 2022-03-21 00:08:02+08:00 2022-03-21 00:08:54+08:00,              overlap_minute
0 2022-03-21 00:00:00+08:00
1 2022-03-21 00:02:00+08:00)```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse loaded dataframes if available; otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv', low_memory=False)
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

try:
    df_err
except NameError:
    df_err = pd.read_csv('error_logs.csv', low_memory=False)
    df_err['ts'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure ts columns exist and are timezone-aware
if 'ts' not in df_trace.columns:
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
if 'ts' not in df_err.columns:
    df_err['ts'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Candidate minutes (UTC+8)
minute_00 = pd.Timestamp('2022-03-21 00:00:00', tz=tz)
minute_02 = pd.Timestamp('2022-03-21 00:02:00', tz=tz)
candidate_minutes = [minute_00, minute_02]

# Frontend list
frontends = ["frontend-0","frontend-1","frontend-2"]

# 1) Trace inspection per minute
# Condition: ts == minute AND (cmdb_id in frontends OR trace_name contains "from_frontend" OR trace_name contains "to_" with 'frontend' involved)
def select_trace_for_minute(minute):
    # exact-minute rows (trace timestamps are floored to minute)
    df_min = df_trace[df_trace['ts'] == minute]
    # selection predicate
    pred_cmdb = df_min['cmdb_id'].isin(frontends)
    name = df_min['trace_name'].astype(str)
    pred_from = name.str.contains('from_frontend', na=False)
    pred_to_frontend = name.str.contains('to_frontend', na=False)
    # also include names that contain 'to_' and 'frontend' anywhere (conservative)
    pred_to_with_frontend = name.str.contains('to_.*frontend', regex=True, na=False) | (
        name.str.contains('to_', na=False) & name.str.contains('frontend', na=False)
    )
    pred = pred_cmdb | pred_from | pred_to_frontend | pred_to_with_frontend
    sel = df_min[pred].copy()
    # Select requested columns and sort by value desc, limit top 20
    if not sel.empty:
        sel = sel[['ts','cmdb_id','trace_name','value']].sort_values('value', ascending=False).head(20).reset_index(drop=True)
    else:
        sel = pd.DataFrame(columns=['ts','cmdb_id','trace_name','value'])
    return sel

trace_top_00 = select_trace_for_minute(minute_00)
trace_top_02 = select_trace_for_minute(minute_02)

# 2) Error-log message details per minute for frontends
def error_messages_for_minute(minute):
    # rows with ts in the minute (any second inside minute)
    start = minute
    end = minute + pd.Timedelta(minutes=1)
    df_minute = df_err[(df_err['ts'] >= start) & (df_err['ts'] < end) & (df_err['cmdb_id'].isin(frontends))].copy()
    if df_minute.empty:
        return pd.DataFrame(columns=['minute','cmdb_id','message','count','earliest_ts'])
    # group by cmdb_id and message
    grp = (df_minute.groupby(['cmdb_id','message'])
           .agg(count=('message','size'), earliest_ts=('ts','min'))
           .reset_index())
    # For each (cmdb_id) keep top 20 messages by count
    top_per_cmdb = (grp.sort_values(['cmdb_id','count'], ascending=[True, False])
                    .groupby('cmdb_id', group_keys=False)
                    .apply(lambda g: g.head(20))
                    .reset_index(drop=True))
    # add minute column
    top_per_cmdb.insert(0, 'minute', minute)
    return top_per_cmdb[['minute','cmdb_id','message','count','earliest_ts']]

err_msgs_00 = error_messages_for_minute(minute_00)
err_msgs_02 = error_messages_for_minute(minute_02)

# 3) Small summary table per minute: total_error_rows_in_minute and per-frontend counts
def error_summary_minute(minute):
    start = minute
    end = minute + pd.Timedelta(minutes=1)
    df_minute = df_err[(df_err['ts'] >= start) & (df_err['ts'] < end)]
    total = len(df_minute)
    counts = df_minute[df_minute['cmdb_id'].isin(frontends)].groupby('cmdb_id').size().reindex(frontends, fill_value=0)
    summary = pd.DataFrame([{
        'minute': minute,
        'total_error_rows_in_minute': int(total),
        'frontend-0': int(counts.get('frontend-0',0)),
        'frontend-1': int(counts.get('frontend-1',0)),
        'frontend-2': int(counts.get('frontend-2',0))
    }])
    return summary

err_summary_00 = error_summary_minute(minute_00)
err_summary_02 = error_summary_minute(minute_02)

# Final compact outputs:
# trace_top_00, trace_top_02
# err_msgs_00, err_msgs_02
# err_summary_00, err_summary_02

trace_top_00, trace_top_02, err_msgs_00, err_msgs_02, err_summary_00, err_summary_02

```
Out[6]:
```
Summary for the two candidate minutes (UTC+8):

Minute 2022-03-21 00:00:00
- Trace top rows (by value, top examples):
  - frontend-0 trace.self.row_count = 337
  - frontend-2 trace.self.row_count = 267
  - frontend-1 trace.self.row_count = 259
  - Several downstream call counts from frontends to productcatalogservice and currencyservice (~43–44) and frontend->currencyservice (44).
- Error-log messages (frontends only):
  - All frontends show the same dominant message: "severity: warning, message: failed to retrieve..." 
  - Counts in-minute: frontend-0 = 10, frontend-1 = 7, frontend-2 = 6 (total 23 errors).
  - Earliest error timestamps in that minute: frontend-0 00:00:11, frontend-1 00:00:12, frontend-2 00:00:05.
- Minute-level summary: total_error_rows_in_minute = 23; per-frontend: frontend-0:10, frontend-1:7, frontend-2:6.

Minute 2022-03-21 00:02:00
- Trace top rows (by value, top examples):
  - frontend-0 trace.self.row_count = 386
  - frontend-2 trace.self.row_count = 273
  - frontend-1 trace.self.row_count = 251
  - Larger "from_frontend2-0" counts on productcatalogservice2-0 (177) and currencyservice2-0 (128); multiple frontend->productcatalogservice and frontend->currencyservice call counts around 58–39.
- Error-log messages (frontends only):
  - Same dominant message "severity: warning, message: failed to retrieve..." 
  - Counts in-minute: frontend-0 = 10, frontend-1 = 6, frontend-2 = 7 (total 23 errors).
  - Earliest error timestamps in that minute: frontend-0 00:02:10, frontend-1 00:02:16, frontend-2 00:02:13.
- Minute-level summary: total_error_rows_in_minute = 23; per-frontend: frontend-0:10, frontend-1:6, frontend-2:7.

Interpretation / quick conclusion:
- Both minutes show high frontend activity (trace.self.row_count peaks) together with repeated frontend error logs containing the same warning message.
- The error bursts are concentrated in the three frontend nodes and are temporally aligned with high outgoing trace counts from frontends to backend services (notably productcatalogservice and currencyservice), suggesting the frontends experienced issues that coincide with elevated request traffic to downstream services.

The original code execution output of IPython Kernel is also provided below for reference:

(                          ts                   cmdb_id                                  trace_name  value
0  2022-03-21 00:00:00+08:00                frontend-0                        trace.self.row_count  337.0
1  2022-03-21 00:00:00+08:00                frontend-2                        trace.self.row_count  267.0
2  2022-03-21 00:00:00+08:00                frontend-1                        trace.self.row_count  259.0
3  2022-03-21 00:00:00+08:00        currencyservice2-0            trace.from_frontend2-0.row_count  142.0
4  2022-03-21 00:00:00+08:00  productcatalogservice2-0            trace.from_frontend2-0.row_count  136.0
5  2022-03-21 00:00:00+08:00                frontend-0        trace.to_currencyservice-0.row_count   44.0
6  2022-03-21 00:00:00+08:00         currencyservice-0             trace.from_frontend-0.row_count   44.0
7  2022-03-21 00:00:00+08:00   productcatalogservice-2             trace.from_frontend-0.row_count   43.0
8  2022-03-21 00:00:00+08:00         currencyservice-1             trace.from_frontend-0.row_count   43.0
9  2022-03-21 00:00:00+08:00   productcatalogservice-1             trace.from_frontend-0.row_count   43.0
10 2022-03-21 00:00:00+08:00                frontend-0        trace.to_currencyservice-1.row_count   43.0
11 2022-03-21 00:00:00+08:00                frontend-0  trace.to_productcatalogservice-0.row_count   43.0
12 2022-03-21 00:00:00+08:00         currencyservice-2             trace.from_frontend-0.row_count   43.0
13 2022-03-21 00:00:00+08:00   productcatalogservice-0             trace.from_frontend-0.row_count   43.0
14 2022-03-21 00:00:00+08:00                frontend-0  trace.to_productcatalogservice-2.row_count   43.0
15 2022-03-21 00:00:00+08:00                frontend-0  trace.to_productcatalogservice-1.row_count   43.0
16 2022-03-21 00:00:00+08:00                frontend-0        trace.to_currencyservice-2.row_count   43.0
17 2022-03-21 00:00:00+08:00                frontend-0                   trace.from_root.row_count   35.0
18 2022-03-21 00:00:00+08:00   productcatalogservice-0             trace.from_frontend-2.row_count   35.0
19 2022-03-21 00:00:00+08:00                frontend-2  trace.to_productcatalogservice-0.row_count   35.0,                           ts                   cmdb_id                                  trace_name  value
0  2022-03-21 00:02:00+08:00                frontend-0                        trace.self.row_count  386.0
1  2022-03-21 00:02:00+08:00                frontend-2                        trace.self.row_count  273.0
2  2022-03-21 00:02:00+08:00                frontend-1                        trace.self.row_count  251.0
3  2022-03-21 00:02:00+08:00  productcatalogservice2-0            trace.from_frontend2-0.row_count  177.0
4  2022-03-21 00:02:00+08:00        currencyservice2-0            trace.from_frontend2-0.row_count  128.0
5  2022-03-21 00:02:00+08:00   productcatalogservice-2             trace.from_frontend-0.row_count   58.0
6  2022-03-21 00:02:00+08:00   productcatalogservice-1             trace.from_frontend-0.row_count   58.0
7  2022-03-21 00:02:00+08:00                frontend-0  trace.to_productcatalogservice-0.row_count   58.0
8  2022-03-21 00:02:00+08:00                frontend-0  trace.to_productcatalogservice-2.row_count   58.0
9  2022-03-21 00:02:00+08:00                frontend-0  trace.to_productcatalogservice-1.row_count   58.0
10 2022-03-21 00:02:00+08:00   productcatalogservice-0             trace.from_frontend-0.row_count   58.0
11 2022-03-21 00:02:00+08:00                frontend-1  trace.to_productcatalogservice-1.row_count   40.0
12 2022-03-21 00:02:00+08:00   productcatalogservice-1             trace.from_frontend-1.row_count   40.0
13 2022-03-21 00:02:00+08:00   productcatalogservice-2             trace.from_frontend-1.row_count   39.0
14 2022-03-21 00:02:00+08:00   productcatalogservice-0             trace.from_frontend-1.row_count   39.0
15 2022-03-21 00:02:00+08:00         currencyservice-0             trace.from_frontend-0.row_count   39.0
16 2022-03-21 00:02:00+08:00                frontend-1  trace.to_productcatalogservice-2.row_count   39.0
17 2022-03-21 00:02:00+08:00                frontend-0        trace.to_currencyservice-2.row_count   39.0
18 2022-03-21 00:02:00+08:00                frontend-0        trace.to_currencyservice-0.row_count   39.0
19 2022-03-21 00:02:00+08:00                frontend-0        trace.to_currencyservice-1.row_count   39.0,                      minute     cmdb_id                                            message  count               earliest_ts
0 2022-03-21 00:00:00+08:00  frontend-0  severity: warning, message: failed to retrieve...     10 2022-03-21 00:00:11+08:00
1 2022-03-21 00:00:00+08:00  frontend-1  severity: warning, message: failed to retrieve...      7 2022-03-21 00:00:12+08:00
2 2022-03-21 00:00:00+08:00  frontend-2  severity: warning, message: failed to retrieve...      6 2022-03-21 00:00:05+08:00,                      minute     cmdb_id                                            message  count               earliest_ts
0 2022-03-21 00:02:00+08:00  frontend-0  severity: warning, message: failed to retrieve...     10 2022-03-21 00:02:10+08:00
1 2022-03-21 00:02:00+08:00  frontend-1  severity: warning, message: failed to retrieve...      6 2022-03-21 00:02:16+08:00
2 2022-03-21 00:02:00+08:00  frontend-2  severity: warning, message: failed to retrieve...      7 2022-03-21 00:02:13+08:00,                      minute  total_error_rows_in_minute  frontend-0  frontend-1  frontend-2
0 2022-03-21 00:00:00+08:00                          23          10           7           6,                      minute  total_error_rows_in_minute  frontend-0  frontend-1  frontend-2
0 2022-03-21 00:02:00+08:00                          23          10           6           7)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse dataframes if present; otherwise load and prepare tz-aware 'ts' columns
try:
    df_err
except NameError:
    df_err = pd.read_csv('error_logs.csv', low_memory=False)
    df_err['ts'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    if 'ts' not in df_err.columns:
        df_err['ts'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv', low_memory=False)
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    if 'ts' not in df_trace.columns:
        df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Candidate minutes (UTC+8)
minute_00 = pd.Timestamp('2022-03-21 00:00:00', tz=tz)
minute_02 = pd.Timestamp('2022-03-21 00:02:00', tz=tz)
candidate_minutes = [minute_00, minute_02]

frontends = ["frontend-0","frontend-1","frontend-2"]

# 1) error_logs: for each (minute, frontend) up to 20 unique messages with counts and earliest ts in minute
err_rows = []
for minute in candidate_minutes:
    start = minute
    end = minute + pd.Timedelta(minutes=1)
    df_min = df_err[(df_err['ts'] >= start) & (df_err['ts'] < end) & (df_err['cmdb_id'].isin(frontends))].copy()
    if df_min.empty:
        continue
    grp = (
        df_min
        .groupby(['cmdb_id','message'], dropna=False)
        .agg(count=('message','size'), earliest_ts_in_minute=('ts','min'))
        .reset_index()
        .sort_values(['cmdb_id','count','earliest_ts_in_minute'], ascending=[True, False, True])
    )
    # keep up to 20 messages per (minute, cmdb_id)
    grp_limited = grp.groupby('cmdb_id', group_keys=False).apply(lambda g: g.head(20)).reset_index(drop=True)
    grp_limited.insert(0, 'minute', minute)  # add minute column
    # rename columns to requested names
    grp_limited = grp_limited.rename(columns={'message':'message','count':'count','earliest_ts_in_minute':'earliest_ts_in_minute'})
    err_rows.append(grp_limited[['minute','cmdb_id','message','count','earliest_ts_in_minute']])

if err_rows:
    err_messages_all = pd.concat(err_rows, ignore_index=True)
    # Cap overall rows to at most 60 as requested
    err_messages_limited = err_messages_all.head(60).copy()
else:
    err_messages_limited = pd.DataFrame(columns=['minute','cmdb_id','message','count','earliest_ts_in_minute'])

# 2) trace.csv: select rows where ts equals candidate minute AND cmdb_id in frontends or startswith productcatalogservice/currencyservice
# and trace_name contains any of the target substrings (duration, duration_mean, duration_p95, error_rate)
patterns = ['duration','duration_mean','duration_p95','error_rate']
name_mask_regex = '|'.join([p for p in patterns])
trace_sel_rows = []
for minute in candidate_minutes:
    df_min = df_trace[df_trace['ts'] == minute].copy()
    if df_min.empty:
        continue
    cmdb_mask = (
        df_min['cmdb_id'].isin(frontends) |
        df_min['cmdb_id'].str.startswith('productcatalogservice', na=False) |
        df_min['cmdb_id'].str.startswith('currencyservice', na=False)
    )
    name_mask = df_min['trace_name'].astype(str).str.contains(name_mask_regex, case=False, na=False)
    sel = df_min[cmdb_mask & name_mask].copy()
    if not sel.empty:
        sel = sel[['ts','cmdb_id','trace_name','value']].sort_values('value', ascending=False)
        trace_sel_rows.append(sel)
if trace_sel_rows:
    trace_selected = pd.concat(trace_sel_rows, ignore_index=True).sort_values('value', ascending=False).head(50).reset_index(drop=True)
else:
    trace_selected = pd.DataFrame(columns=['ts','cmdb_id','trace_name','value'])

# Final compact outputs (DataFrames)
err_messages_limited, trace_selected

```
Out[7]:
```
Summary for the two target minutes (UTC+8):

1) error_logs.csv (messages per frontend, up to 20 messages each)
- Minute 2022-03-21 00:00:
  - frontend-0: 10 occurrences of the same message ("severity: warning, message: failed to retrieve..."), earliest at 00:00:11.
  - frontend-1: 7 occurrences of the same message, earliest at 00:00:12.
  - frontend-2: 6 occurrences of the same message, earliest at 00:00:05.
- Minute 2022-03-21 00:02:
  - frontend-0: 10 occurrences of the same message, earliest at 00:02:10.
  - frontend-1: 6 occurrences of the same message, earliest at 00:02:16.
  - frontend-2: 7 occurrences of the same message, earliest at 00:02:13.
- Observation: the error activity in both minutes is dominated by a single repeated warning (“failed to retrieve...”) on all three frontends.

2) trace.csv (selected duration/error-rate metrics; top 50 rows by value)
- Highest values (examples from the top of the list):
  - 00:02 — frontend-2 — trace.to_checkoutservice-1.duration_mean = 0.083544 (also duration_p95 same value).
  - 00:00 / 00:02 — frontend-2 / frontend-1 — trace.from_root.duration_p95 around 0.067–0.068 and 0.055–0.068.
  - Multiple frontend -> productcatalogservice and frontend -> currencyservice trace.duration_p95 entries appear (values in the ~0.005–0.05 range).
- General pattern: duration and p95 metrics are small (fractions of a second). The largest observed trace duration in these minutes is ~0.0835s (frontend-2 to checkoutservice at 00:02). Many p95 duration entries for frontend→productcatalogservice / currencyservice are present across both minutes.

3) Compact conclusion
- Both minutes (00:00 and 00:02) show concentrated frontend error logs (same warning message) and substantive trace activity from frontends to backend services.
- Notable trace-level anomaly at 00:02: frontend-2 shows the largest duration values (to checkoutservice), coinciding with a 10-count error spike on frontend-0 and elevated errors on frontend-1/2.
- These aligned observations (repeated frontend warnings + elevated per-call durations from frontends) point to a frontend-centered failure pattern at those minutes.

The original code execution output of IPython Kernel is also provided below for reference:

(                     minute     cmdb_id                                            message  count     earliest_ts_in_minute
0 2022-03-21 00:00:00+08:00  frontend-0  severity: warning, message: failed to retrieve...     10 2022-03-21 00:00:11+08:00
1 2022-03-21 00:00:00+08:00  frontend-1  severity: warning, message: failed to retrieve...      7 2022-03-21 00:00:12+08:00
2 2022-03-21 00:00:00+08:00  frontend-2  severity: warning, message: failed to retrieve...      6 2022-03-21 00:00:05+08:00
3 2022-03-21 00:02:00+08:00  frontend-0  severity: warning, message: failed to retrieve...     10 2022-03-21 00:02:10+08:00
4 2022-03-21 00:02:00+08:00  frontend-1  severity: warning, message: failed to retrieve...      6 2022-03-21 00:02:16+08:00
5 2022-03-21 00:02:00+08:00  frontend-2  severity: warning, message: failed to retrieve...      7 2022-03-21 00:02:13+08:00,                           ts                   cmdb_id                                     trace_name     value
0  2022-03-21 00:02:00+08:00                frontend-2       trace.to_checkoutservice-1.duration_mean  0.083544
1  2022-03-21 00:02:00+08:00                frontend-2        trace.to_checkoutservice-1.duration_p95  0.083544
2  2022-03-21 00:00:00+08:00                frontend-2                   trace.from_root.duration_p95  0.068345
3  2022-03-21 00:02:00+08:00                frontend-2                   trace.from_root.duration_p95  0.067931
4  2022-03-21 00:02:00+08:00                frontend-1                   trace.from_root.duration_p95  0.055866
..                       ...                       ...                                            ...       ...
45 2022-03-21 00:00:00+08:00                frontend-2  trace.to_productcatalogservice-0.duration_p95  0.005605
46 2022-03-21 00:00:00+08:00                frontend-2  trace.to_productcatalogservice-1.duration_p95  0.005462
47 2022-03-21 00:00:00+08:00   productcatalogservice-1             trace.from_frontend-2.duration_p95  0.005462
48 2022-03-21 00:02:00+08:00  productcatalogservice2-0     trace.from_checkoutservice2-0.duration_p95  0.005457
49 2022-03-21 00:00:00+08:00                frontend-0  trace.to_productcatalogservice-1.duration_p95  0.005427

[50 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

# Reuse existing df_trace if present; otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv', low_memory=False)

# Ensure timezone-aware 'ts' column in UTC+8
tz = pytz.timezone('Asia/Shanghai')
if 'ts' not in df_trace.columns:
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz-aware and converted
    df_trace['ts'] = pd.to_datetime(df_trace['ts']).dt.tz_localize('UTC').dt.tz_convert(tz) if df_trace['ts'].dtype == object else df_trace['ts']

# Define candidate minutes explicitly (UTC+8)
minute_00 = pd.Timestamp('2022-03-21 00:00:00', tz=tz)
minute_02 = pd.Timestamp('2022-03-21 00:02:00', tz=tz)

# Selection criteria for cmdb_id
frontends = ["frontend-0","frontend-1","frontend-2"]
cmdb_mask = (
    df_trace['cmdb_id'].isin(frontends) |
    df_trace['cmdb_id'].str.startswith('productcatalogservice', na=False) |
    df_trace['cmdb_id'].str.startswith('currencyservice', na=False) |
    df_trace['cmdb_id'].str.startswith('checkoutservice', na=False)
)

# Selection criteria for trace_name (contains any of the requested substrings)
pattern = r'(error_rate|duration_mean|duration_p95|duration_p50|duration)'
name_mask = df_trace['trace_name'].astype(str).str.contains(pattern, case=False, na=False)

# Filter to the two exact minutes and apply masks
time_mask = (df_trace['ts'] == minute_00) | (df_trace['ts'] == minute_02)
sel = df_trace[time_mask & cmdb_mask & name_mask].copy()

# Select and rename columns, sort by value desc, limit to top 50
trace_filtered_top50 = (
    sel[['ts','cmdb_id','trace_name','value']]
    .rename(columns={'ts':'timestamp'})
    .sort_values(by='value', ascending=False)
    .head(50)
    .reset_index(drop=True)
)

trace_filtered_top50

```
Out[8]:
```
Summary (trace metrics at 2022-03-21 00:00 and 00:02 UTC+8):

- Top overall latency observations (top rows by value):
  1. 00:02 — checkoutservice2-0 — trace.from_frontend2-0.duration_mean = 0.091737 s (also duration_p95 same).
  2. 00:02 — frontend-2 — trace.to_checkoutservice-1.duration_p95 = 0.083544 s (duration_mean equal).
  3. 00:02 — checkoutservice-1 — trace.from_frontend-2.duration_p95 = 0.083544 s.
  4. Several frontend -> productcatalogservice / currencyservice and frontend self.duration_p95 entries appear across both minutes (values range from ~0.005–0.08 s).
  5. The largest single value is ~0.0917 s (checkoutservice2-0 at 00:02); many other p95/duration entries cluster around 0.04–0.08 s at 00:02.

- Minute-level pattern:
  - 00:02 shows the highest and most pronounced duration values (including the top two entries).
  - 00:00 contains many duration_p95 entries but generally lower than the 00:02 peaks.

- Interpretation:
  - The highest trace latencies at these two minutes concentrate around frontends calling checkoutservice (and backend services like checkoutservice2-0), with the strongest spike at 00:02.
  - Values are sub-second (tens of milliseconds), but relative elevation at 00:02 (vs 00:00 and other entries) suggests a transient increase in tail latency for checkout-related flows at that minute.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp             cmdb_id                                trace_name     value
0  2022-03-21 00:02:00+08:00  checkoutservice2-0      trace.from_frontend2-0.duration_mean  0.091737
1  2022-03-21 00:02:00+08:00  checkoutservice2-0       trace.from_frontend2-0.duration_p95  0.091737
2  2022-03-21 00:02:00+08:00          frontend-2   trace.to_checkoutservice-1.duration_p95  0.083544
3  2022-03-21 00:02:00+08:00          frontend-2  trace.to_checkoutservice-1.duration_mean  0.083544
4  2022-03-21 00:02:00+08:00   checkoutservice-1        trace.from_frontend-2.duration_p95  0.083544
..                       ...                 ...                                       ...       ...
45 2022-03-21 00:00:00+08:00  currencyservice2-0       trace.from_frontend2-0.duration_p95  0.007458
46 2022-03-21 00:02:00+08:00   checkoutservice-2                   trace.self.duration_p95  0.007129
47 2022-03-21 00:00:00+08:00   checkoutservice-0                   trace.self.duration_p95  0.006822
48 2022-03-21 00:02:00+08:00          frontend-0                   trace.self.duration_p95  0.006790
49 2022-03-21 00:00:00+08:00   checkoutservice-1                   trace.self.duration_p95  0.006779

[50 rows x 4 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz

# Reuse df_metric and tz if present; otherwise load and prepare
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', low_memory=False)

tz = pytz.timezone('Asia/Shanghai')

# Ensure timezone-aware 'ts' column in UTC+8
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # If ts exists but may be naive, coerce safely
    if not pd.api.types.is_datetime64tz_dtype(df_metric['ts']):
        df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Target frontends and minutes
frontends = ["frontend-0","frontend-1","frontend-2"]
minute_00 = pd.Timestamp('2022-03-21 00:00:00', tz=tz)
minute_02 = pd.Timestamp('2022-03-21 00:02:00', tz=tz)

# 1) Compute global P95, P5, full_series_count for (cmdb_id, kpi_name) restricted to target frontends
df_metric_front = df_metric[df_metric['cmdb_id'].isin(frontends)].copy()

group_stats = (
    df_metric_front
    .groupby(['cmdb_id','kpi_name'])['value']
    .agg(full_series_count='count',
         P95=lambda x: float(x.quantile(0.95)),
         P5=lambda x: float(x.quantile(0.05)))
    .reset_index()
)

# 2) Extract values at the two target minutes (exact match)
# Keep only one value per (cmdb_id,kpi_name,ts) if duplicates exist (take first)
df_minutes = df_metric_front[df_metric_front['ts'].isin([minute_00, minute_02])].copy()
df_minutes = df_minutes.sort_values('timestamp')  # stable order
df_minutes_unique = df_minutes.drop_duplicates(subset=['cmdb_id','kpi_name','ts'], keep='first')

# Pivot extracted minute values into columns
val_00 = df_minutes_unique[df_minutes_unique['ts'] == minute_00][['cmdb_id','kpi_name','value']].rename(columns={'value':'value_at_00:00'})
val_02 = df_minutes_unique[df_minutes_unique['ts'] == minute_02][['cmdb_id','kpi_name','value']].rename(columns={'value':'value_at_00:02'})

# Merge stats with extracted values
merged = group_stats.merge(val_00, on=['cmdb_id','kpi_name'], how='left').merge(val_02, on=['cmdb_id','kpi_name'], how='left')

# 3) Compute boolean flags for each extracted value vs global thresholds (use NA when value missing)
def ge_flag(val, thr):
    if pd.isna(val):
        return pd.NA
    return bool(val >= thr)

def le_flag(val, thr):
    if pd.isna(val):
        return pd.NA
    return bool(val <= thr)

merged['value_at_00:00_ge_P95'] = merged.apply(lambda r: ge_flag(r.get('value_at_00:00'), r['P95']), axis=1)
merged['value_at_00:00_le_P5'] = merged.apply(lambda r: le_flag(r.get('value_at_00:00'), r['P5']), axis=1)
merged['value_at_00:02_ge_P95'] = merged.apply(lambda r: ge_flag(r.get('value_at_00:02'), r['P95']), axis=1)
merged['value_at_00:02_le_P5'] = merged.apply(lambda r: le_flag(r.get('value_at_00:02'), r['P5']), axis=1)

# 4) Prepare final compact table, sorted by full_series_count desc, then kpi_name asc; limit to 50 rows
final_cols = [
    'cmdb_id','kpi_name','full_series_count','P95','P5',
    'value_at_00:00','value_at_00:00_ge_P95','value_at_00:00_le_P5',
    'value_at_00:02','value_at_00:02_ge_P95','value_at_00:02_le_P5'
]
final = merged.sort_values(by=['full_series_count','kpi_name'], ascending=[False, True])[final_cols].head(50).reset_index(drop=True)

# Display compact final result
final

```
Out[9]:
```
Summary (metric.csv for frontend-0/1/2 at 2022-03-21 00:00 and 00:02 UTC+8):

- Scope and data:
  - Analysis covered (cmdb_id, kpi_name) pairs for frontend-0, frontend-1, frontend-2.
  - The returned table shows the top 50 pairs (sorted by full_series_count). Most pairs have full_series_count = 30 (data covers 30 minutes).

- Key findings:
  - No meaningful spikes above historical P95 were observed at the two target minutes. For example:
    - frontend-0, container.node-5.container_cpu_cfs_periods: P95 = 111.225; value_at_00:00 = 101.5 and value_at_00:02 = 106.0 (both below P95).
    - frontend-1 and frontend-2 CPU-related values at those minutes are also below their P95 values.
  - Many KPIs are constant zeros across the series (P95 = P5 = 0). For those metrics the flags value_ge_P95 and value_le_P5 are True because the minute values equal the threshold (0). These “True” flags reflect equality with a zero baseline, not an upward spike.
  - A mix of metrics (CPU, filesystem, throttling, I/O counters) were present; none show a clear outlier spike (value > P95) at 00:00 or 00:02 in the top-50 results.

- Interpretation:
  - Frontend metrics at the two examined minutes do not exhibit values exceeding their historical P95 (i.e., no clear metric-level spike). The apparent “anomalies” are mostly zero-valued KPIs where P95==P5==0 and the minute values equal zero.
  - This aligns with trace/error observations: frontends showed many repeated error logs and elevated trace activity, but the standard host/container KPIs for these frontends do not show concurrent metric spikes at those exact minutes.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  full_series_count      P95      P5  value_at_00:00  value_at_00:00_ge_P95  value_at_00:00_le_P5  value_at_00:02  value_at_00:02_ge_P95  value_at_00:02_le_P5
0   frontend-0         container.node-5.container_cpu_cfs_periods                 30  111.225  72.175           101.5                  False                 False           106.0                  False                 False
1   frontend-1         container.node-5.container_cpu_cfs_periods                 30   75.100  56.000            70.5                  False                 False            58.5                  False                 False
2   frontend-2         container.node-5.container_cpu_cfs_periods                 30   79.200  53.175            71.0                  False                 False            63.0                  False                 False
3   frontend-0  container.node-5.container_cpu_cfs_throttled_p...                 30    0.000   0.000             0.0                   True                  True             0.0                   True                  True
4   frontend-1  container.node-5.container_cpu_cfs_throttled_p...                 30    0.000   0.000             0.0                   True                  True             0.0                   True                  True
..         ...                                                ...                ...      ...     ...             ...                    ...                   ...             ...                    ...                   ...
45  frontend-0       container.node-5.container_fs_reads./dev/vda                 30    0.000   0.000             0.0                   True                  True             0.0                   True                  True
46  frontend-1       container.node-5.container_fs_reads./dev/vda                 30    0.000   0.000             0.0                   True                  True             0.0                   True                  True
47  frontend-2       container.node-5.container_fs_reads./dev/vda                 30    0.000   0.000             0.0                   True                  True             0.0                   True                  True
48  frontend-0      container.node-5.container_fs_reads./dev/vda1                 30    0.000   0.000             0.0                   True                  True             0.0                   True                  True
49  frontend-1      container.node-5.container_fs_reads./dev/vda1                 30    0.000   0.000             0.0                   True                  True             0.0                   True                  True

[50 rows x 11 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse df_trace and candidate minute variables if present; otherwise load/define
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv', low_memory=False)

# ensure timezone-aware 'ts' column in UTC+8
if 'ts' not in df_trace.columns:
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    if not pd.api.types.is_datetime64tz_dtype(df_trace['ts']):
        df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# define target minutes (reuse if available)
try:
    minute_00, minute_02
except NameError:
    minute_00 = pd.Timestamp('2022-03-21 00:00:00', tz=tz)
    minute_02 = pd.Timestamp('2022-03-21 00:02:00', tz=tz)

# Filter: exact timestamp equals one of the two minutes AND cmdb_id startswith 'checkoutservice'
time_mask = (df_trace['ts'] == minute_00) | (df_trace['ts'] == minute_02)
cmdb_mask = df_trace['cmdb_id'].astype(str).str.startswith('checkoutservice', na=False)
name_mask = df_trace['trace_name'].astype(str).str.contains('error', case=False, na=False)

sel = df_trace[time_mask & cmdb_mask & name_mask].copy()

# Prepare result: select requested columns, rename ts to timestamp, sort by value desc, limit 50
if sel.empty:
    checkout_error_traces = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value'])
else:
    checkout_error_traces = (
        sel[['ts','cmdb_id','trace_name','value']]
        .rename(columns={'ts':'timestamp'})
        .sort_values(by='value', ascending=False)
        .head(50)
        .reset_index(drop=True)
    )

checkout_error_traces

```
Out[10]:
```
Summary (checkoutservice error-rate traces at 00:00 and 00:02 UTC+8):

- Matches found: 50 trace rows (limited to top 50) for cmdb_id starting with "checkoutservice" at the two target minutes.
- All returned trace_name entries are error_rate-related (examples):
  - 2022-03-21 00:00 — checkoutservice-0: trace.from_frontend-2.error_rate, trace.from_root.error_rate, trace.self.error_rate, trace.to_cartservice-0.error_rate, ...
  - 2022-03-21 00:02 — checkoutservice-1 / checkoutservice-2: many trace.to_* and trace.from_frontend-*.error_rate entries.
- All values are 0.0 (no observed error rates at those minutes for the checkoutservice traces).
- Interpretation: checkoutservice components show no error-rate signals at 00:00 or 00:02 (error_rate = 0), so they are unlikely to be the immediate source of the frontend error messages observed earlier.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp            cmdb_id                             trace_name  value
0  2022-03-21 00:00:00+08:00  checkoutservice-0       trace.from_frontend-2.error_rate    0.0
1  2022-03-21 00:00:00+08:00  checkoutservice-0             trace.from_root.error_rate    0.0
2  2022-03-21 00:00:00+08:00  checkoutservice-0                  trace.self.error_rate    0.0
3  2022-03-21 00:00:00+08:00  checkoutservice-0      trace.to_cartservice-0.error_rate    0.0
4  2022-03-21 00:00:00+08:00  checkoutservice-0      trace.to_cartservice-2.error_rate    0.0
..                       ...                ...                                    ...    ...
45 2022-03-21 00:02:00+08:00  checkoutservice-1  trace.to_shippingservice-1.error_rate    0.0
46 2022-03-21 00:02:00+08:00  checkoutservice-1  trace.to_shippingservice-2.error_rate    0.0
47 2022-03-21 00:02:00+08:00  checkoutservice-2       trace.from_frontend-1.error_rate    0.0
48 2022-03-21 00:02:00+08:00  checkoutservice-2       trace.from_frontend-2.error_rate    0.0
49 2022-03-21 00:02:00+08:00  checkoutservice-2             trace.from_root.error_rate    0.0

[50 rows x 4 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[11]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_metric if available; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', low_memory=False)

# Ensure timezone-aware 'ts' column (UTC -> convert to Asia/Shanghai)
if 'ts' not in df_metric.columns or not pd.api.types.is_datetime64tz_dtype(df_metric['ts']):
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Target cmdb_id prefix and minutes
cmdb_prefix = 'checkoutservice'
mask_checkout = df_metric['cmdb_id'].astype(str).str.startswith(cmdb_prefix, na=False)

minute_00 = pd.Timestamp('2022-03-21 00:00:00', tz=tz)
minute_02 = pd.Timestamp('2022-03-21 00:02:00', tz=tz)

# 1) Global thresholds (P95, P5) and full_series_count for checkoutservice pairs (computed on entire file)
group_stats_checkout = (
    df_metric[mask_checkout]
    .groupby(['cmdb_id','kpi_name'])['value']
    .agg(
        full_series_count='count',
        P95=lambda x: float(x.quantile(0.95)),
        P5=lambda x: float(x.quantile(0.05))
    )
    .reset_index()
)

# 2) Extract values at exact minutes (if present)
# Keep first occurrence if duplicates
vals_minutes = df_metric[mask_checkout & df_metric['ts'].isin([minute_00, minute_02])].copy()
vals_minutes = vals_minutes.sort_values('timestamp').drop_duplicates(subset=['cmdb_id','kpi_name','ts'], keep='first')

val_00 = vals_minutes[vals_minutes['ts'] == minute_00][['cmdb_id','kpi_name','value']].rename(columns={'value':'value_at_00:00'})
val_02 = vals_minutes[vals_minutes['ts'] == minute_02][['cmdb_id','kpi_name','value']].rename(columns={'value':'value_at_00:02'})

# 3) Merge stats with extracted minute values
merged = group_stats_checkout.merge(val_00, on=['cmdb_id','kpi_name'], how='left').merge(val_02, on=['cmdb_id','kpi_name'], how='left')

# Flags: value >= P95, value <= P5 (NA when value missing)
def ge_flag(val, thr):
    if pd.isna(val):
        return pd.NA
    return bool(val >= thr)

def le_flag(val, thr):
    if pd.isna(val):
        return pd.NA
    return bool(val <= thr)

merged['value_at_00:00_ge_P95'] = merged.apply(lambda r: ge_flag(r.get('value_at_00:00'), r['P95']), axis=1)
merged['value_at_00:00_le_P5'] = merged.apply(lambda r: le_flag(r.get('value_at_00:00'), r['P5']), axis=1)
merged['value_at_00:02_ge_P95'] = merged.apply(lambda r: ge_flag(r.get('value_at_00:02'), r['P95']), axis=1)
merged['value_at_00:02_le_P5'] = merged.apply(lambda r: le_flag(r.get('value_at_00:02'), r['P5']), axis=1)

# 4) Rank by absolute deviation of value_at_00:02 from P95 (desc) to surface likely anomalies at 00:02
merged['abs_dev_02'] = (merged['value_at_00:02'] - merged['P95']).abs()
merged['abs_dev_02'] = merged['abs_dev_02'].fillna(0.0)

# Prepare final compact table and limit to top 50
final_cols = [
    'cmdb_id','kpi_name','full_series_count','P95','P5',
    'value_at_00:00','value_at_00:00_ge_P95','value_at_00:00_le_P5',
    'value_at_00:02','value_at_00:02_ge_P95','value_at_00:02_le_P5','abs_dev_02'
]
final_result = merged.sort_values(by=['abs_dev_02','full_series_count','kpi_name'], ascending=[False, False, True])[final_cols].head(50).reset_index(drop=True)

# Display the compact result (drop helper abs_dev_02 from view if not needed; keep it to show ranking relevance)
final_result

```
Out[11]:
```
Summary (checkoutservice metrics, ranked by deviation at 2022-03-21 00:02 UTC+8):

- Overview:
  - Analysis covered checkoutservice* nodes (checkoutservice-0/1/2 and checkoutservice2-0). Table limited to top 50 (ranked by absolute deviation of value_at_00:02 from P95).
  - Each row shows full-series P95/P5 and values at 00:00 and 00:02 with flags indicating >=P95 or <=P5 when present.

- Key findings (high-level highlights from the top-ranked rows):
  1. checkoutservice2-0 — container_memory_rss:
     - P95 ≈ 15,573,610, P5 ≈ 14,890,800.
     - value_at_00:00 = 15,052,800; value_at_00:02 = 14,718,976.
     - At 00:02 the RSS dropped below P5 (value_at_00:02_le_P5 = True). This is the largest absolute deviation in the result set (a significant drop).
  2. checkoutservice-2 / checkoutservice-0 — container_memory_rss:
     - Both show deviations at 00:02 but remain near historical ranges (values slightly below or above P95 but not extreme like checkoutservice2-0).
  3. Several mesh/istio related metrics (e.g., mesh.source.checkoutservice.jaeger-collector.*, mesh.source.checkoutservice.cartservice.istio_*, etc.):
     - Some mesh metrics jumped at 00:02 (value_at_00:02 > P95), e.g., an istio/mesh source metric went from 0 at 00:00 to a large value at 00:02 (flag value_at_00:02_ge_P95 = True).
  4. container memory failure metrics on checkoutservice-2:
     - Metrics such as container_memory_failures.* show non-zero spikes at 00:02 (value_at_00:02 > P95, flags True), indicating increased memory-failure related counts at that minute.
  5. Other notable items:
     - Some checkoutservice mesh/istio counters and service-mesh related KPIs show prominent increases at 00:02 (several rows where value_at_00:02_ge_P95 = True).
     - Many other KPIs have no data at a target minute (NA) or equal baseline values; a subset are zero-valued series where equality to P95/P5 can produce True flags but do not indicate a spike.

- Interpretation / short conclusion:
  - The strongest single signal is a drop in checkoutservice2-0 RSS at 00:02 (below historical P5) — this could indicate memory footprint change or measurement artifact for that instance at that minute.
  - Concurrently, several checkoutservice instances show spikes in mesh/istio and memory-failure counters at 00:02 (values exceeding historical P95). These increases suggest transient operational stress in service-mesh traffic or memory subsystem around 00:02.
  - Combined, the metric-level evidence points to notable activity at 00:02 for checkoutservice components: both a memory RSS drop on checkoutservice2-0 and elevated mesh/istio and memory-failure metrics on other checkoutservice nodes. These minutes merit further focused investigation (logs/traces) for checkout-related flows.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  full_series_count           P95            P5  value_at_00:00  value_at_00:00_ge_P95  value_at_00:00_le_P5  value_at_00:02  value_at_00:02_ge_P95  value_at_00:02_le_P5  abs_dev_02
0   checkoutservice2-0              container.node-5.container_memory_rss                 30  1.557361e+07  1.489080e+07      15052800.0                  False                 False      14718976.0                  False                  True  854630.400
1    checkoutservice-2              container.node-5.container_memory_rss                 30  1.588060e+07  1.493064e+07      15065088.0                  False                 False      15740928.0                  False                 False  139673.600
2    checkoutservice-0              container.node-5.container_memory_rss                 30  1.742766e+07  1.647913e+07      17178624.0                  False                 False      17408000.0                  False                 False   19660.800
3    checkoutservice-0  mesh.source.checkoutservice.jaeger-collector.i...                 30  2.810000e+04  6.048750e+03         14050.0                  False                 False         14050.0                  False                 False   14050.000
4    checkoutservice-2  mesh.source.checkoutservice.jaeger-collector.i...                 30  2.810000e+04  2.362500e+03             0.0                  False                  True         35125.0                   True                 False    7025.000
..                 ...                                                ...                ...           ...           ...             ...                    ...                   ...             ...                    ...                   ...         ...
45   checkoutservice-0  mesh.source.checkoutservice.jaeger-collector.i...                 30  3.980000e+02  7.213750e+01           199.0                  False                 False           199.0                  False                 False     199.000
46   checkoutservice-2  mesh.source.checkoutservice.cartservice.istio_...                 30  7.600000e+02  8.100000e+01             0.0                  False                  True           950.0                   True                 False     190.000
47   checkoutservice-2  container.node-5.container_memory_failures.con...                 30  4.355250e+02  2.272500e+01             0.0                  False                  True           616.0                   True                 False     180.475
48   checkoutservice-2  container.node-5.container_memory_failures.hie...                 30  4.355250e+02  2.272500e+01             0.0                  False                  True           616.0                   True                 False     180.475
49  checkoutservice2-0  mesh.source.checkoutservice2.cartservice2.isti...                 30  9.300000e+02  2.755000e+02           380.0                  False                 False           760.0                  False                 False     170.000

[50 rows x 12 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[12]:

import pandas as pd
import pytz

# Reuse existing df_metric and tz-aware 'ts' if present
tz = pytz.timezone('Asia/Shanghai')
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', low_memory=False)
if 'ts' not in df_metric.columns or not pd.api.types.is_datetime64tz_dtype(df_metric['ts']):
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Targets
frontends = ["frontend-0","frontend-1","frontend-2"]
substrs = ['network','packet','retransmit','receive_err','transmit_err','loss']

# Filter by cmdb_id prefix/list and kpi_name containing any of the substrings (case-insensitive)
mask_cmdb = df_metric['cmdb_id'].isin(frontends)
name_series = df_metric['kpi_name'].astype(str)
mask_name = name_series.str.contains('|'.join(substrs), case=False, na=False)
df_filtered = df_metric[mask_cmdb & mask_name].copy()

# If no matches, return empty-indicator dataframe
if df_filtered.empty:
    empty_cols = [
        'cmdb_id','kpi_name','full_series_count','P95','P5',
        'value_at_00:00','value_at_00:00_ge_P95','value_at_00:00_le_P5',
        'value_at_00:02','value_at_00:02_ge_P95','value_at_00:02_le_P5'
    ]
    final_result = pd.DataFrame(columns=empty_cols)
else:
    # 1) Global P95, P5, full_series_count per (cmdb_id, kpi_name) using entire (filtered) series
    group_stats = (
        df_filtered
        .groupby(['cmdb_id','kpi_name'])['value']
        .agg(full_series_count='count',
             P95=lambda x: float(x.quantile(0.95)),
             P5=lambda x: float(x.quantile(0.05)))
        .reset_index()
    )

    # 2) Extract exact-minute values if present
    minute_00 = pd.Timestamp('2022-03-21 00:00:00', tz=tz)
    minute_02 = pd.Timestamp('2022-03-21 00:02:00', tz=tz)
    vals_minutes = df_filtered[df_filtered['ts'].isin([minute_00, minute_02])].copy()
    vals_minutes = vals_minutes.sort_values('timestamp').drop_duplicates(subset=['cmdb_id','kpi_name','ts'], keep='first')

    val_00 = vals_minutes[vals_minutes['ts'] == minute_00][['cmdb_id','kpi_name','value']].rename(columns={'value':'value_at_00:00'})
    val_02 = vals_minutes[vals_minutes['ts'] == minute_02][['cmdb_id','kpi_name','value']].rename(columns={'value':'value_at_00:02'})

    merged = group_stats.merge(val_00, on=['cmdb_id','kpi_name'], how='left').merge(val_02, on=['cmdb_id','kpi_name'], how='left')

    # 3) Flags (NA when value missing)
    def ge_flag(v, thr):
        return pd.NA if pd.isna(v) else bool(v >= thr)
    def le_flag(v, thr):
        return pd.NA if pd.isna(v) else bool(v <= thr)

    merged['value_at_00:00_ge_P95'] = merged.apply(lambda r: ge_flag(r.get('value_at_00:00'), r['P95']), axis=1)
    merged['value_at_00:00_le_P5'] = merged.apply(lambda r: le_flag(r.get('value_at_00:00'), r['P5']), axis=1)
    merged['value_at_00:02_ge_P95'] = merged.apply(lambda r: ge_flag(r.get('value_at_00:02'), r['P95']), axis=1)
    merged['value_at_00:02_le_P5'] = merged.apply(lambda r: le_flag(r.get('value_at_00:02'), r['P5']), axis=1)

    # 4) Rank by abs deviation of value_at_00:00 from P95 (desc) to surface anomalies at 00:00
    merged['abs_dev_00'] = (merged['value_at_00:00'] - merged['P95']).abs().fillna(0.0)
    final_cols = [
        'cmdb_id','kpi_name','full_series_count','P95','P5',
        'value_at_00:00','value_at_00:00_ge_P95','value_at_00:00_le_P5',
        'value_at_00:02','value_at_00:02_ge_P95','value_at_00:02_le_P5'
    ]
    final_result = merged.sort_values(by=['abs_dev_00','full_series_count','kpi_name'], ascending=[False, False, True])[final_cols].head(50).reset_index(drop=True)

# Display compact bounded result
final_result

```
Out[12]:
```
Summary (network-related metrics for frontend-0/1/2 at 2022-03-21 00:00 and 00:02 UTC+8):

- Scope: 24 (cmdb_id, kpi_name) pairs matched network-related substrings. Results ranked by how far the 00:00 value deviated from the pair’s global P95.

- Top observations:
  - frontend-2:
    - container_network_transmit_packets: P95 ≈ 1016.2. value@00:00 = 839.0 (below P95), value@00:02 = 1076.0 (> P95). => clear transmit-packet surge at 00:02.
    - container_network_receive_packets: P95 ≈ 876.0. value@00:00 = 758.5, value@00:02 = 929.0 (> P95). => receive-packet surge at 00:02 as well.
  - frontend-0:
    - container_network_transmit_packets: P95 ≈ 1500.85. value@00:00 = 1553.5 (> P95) and value@00:02 = 979.0 (below). => transmit spike at 00:00 on frontend-0.
  - frontend-1:
    - container_network_transmit_packets/receive_packets show elevated baseline but did not exceed P95 at these minutes (values at 00:00/00:02 are below P95).
  - Many "transmit_err" / "receive_err" metrics are zero series (P95 = P5 = 0); values at the minutes are 0 (flags True because they equal the baseline), i.e., no packet error counts increased.

- Interpretation:
  - There are short-lived increases in packet transmit/receive rates on the frontends, but they occur at slightly different minutes:
    - frontend-0 shows a transmit peak at 00:00.
    - frontend-2 shows both transmit and receive peaks at 00:02.
  - Crucially, packet error counters remain zero — no evidence of retransmits/receive_err/transmit_err spikes. This points to increased traffic volume/throughput, not network packet errors.

- Conclusion / recommendation:
  - Network throughput bursts on frontends coincide with previously observed trace/error activity. Investigate request traffic patterns (frontends → backends) and upstream load around 00:00–00:02; packet error metrics do not indicate link-level corruption or retransmission issues.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  full_series_count       P95       P5  value_at_00:00  value_at_00:00_ge_P95  value_at_00:00_le_P5  value_at_00:02  value_at_00:02_ge_P95  value_at_00:02_le_P5
0   frontend-2  container.node-5.container_network_transmit_pa...                 30  1016.200  747.350           839.0                  False                 False          1076.0                   True                 False
1   frontend-1  container.node-5.container_network_transmit_pa...                 30  1033.500  708.100           914.0                  False                 False           932.0                  False                 False
2   frontend-2  container.node-5.container_network_receive_pac...                 30   876.025  632.500           758.5                  False                 False           929.0                   True                 False
3   frontend-1  container.node-5.container_network_receive_pac...                 30   894.400  621.725           819.5                  False                 False           818.5                  False                 False
4   frontend-0  container.node-5.container_network_transmit_pa...                 30  1500.850  935.200          1553.5                   True                 False           979.0                  False                 False
..         ...                                                ...                ...       ...      ...             ...                    ...                   ...             ...                    ...                   ...
19  frontend-1  container.node-5.container_network_transmit_er...                 30     0.000    0.000             0.0                   True                  True             0.0                   True                  True
20  frontend-2  container.node-5.container_network_transmit_er...                 30     0.000    0.000             0.0                   True                  True             0.0                   True                  True
21  frontend-0  container.node-5.container_network_transmit_pa...                 30     0.000    0.000             0.0                   True                  True             0.0                   True                  True
22  frontend-1  container.node-5.container_network_transmit_pa...                 30     0.000    0.000             0.0                   True                  True             0.0                   True                  True
23  frontend-2  container.node-5.container_network_transmit_pa...                 30     0.000    0.000             0.0                   True                  True             0.0                   True                  True

[24 rows x 11 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```